In [2]:
import pandas as pd
import numpy as np
import datetime
from epiweeks import Week
import datetime
import random
from datetime import timedelta
import requests
import io
import os

import warnings
warnings.filterwarnings('ignore')

# ILI ensemble forecasts (UnwghtAvg)

In [4]:
forecasts1516 = ['EW01_UnwghtAvg_2016-01-18.csv', 'EW02_UnwghtAvg_2016-01-25.csv', 'EW03_UnwghtAvg_2016-02-01.csv',
                 'EW04_UnwghtAvg_2016-02-08.csv', 'EW05_UnwghtAvg_2016-02-15.csv', 'EW06_UnwghtAvg_2016-02-22.csv',
                 'EW07_UnwghtAvg_2016-02-29.csv', 'EW08_UnwghtAvg_2016-03-07.csv', 'EW09_UnwghtAvg_2016-03-14.csv',
                 'EW10_UnwghtAvg_2016-03-21.csv', 'EW11_UnwghtAvg_2016-03-28.csv', 'EW12_UnwghtAvg_2016-04-04.csv',
                 'EW13_UnwghtAvg_2016-04-11.csv', 'EW14_UnwghtAvg_2016-04-18.csv', 'EW15_UnwghtAvg_2016-04-25.csv',
                 'EW16_UnwghtAvg_2016-05-02.csv', 'EW17_UnwghtAvg_2016-05-09.csv', 'EW18_UnwghtAvg_2016-05-16.csv',
                 'EW42_UnwghtAvg_2015-11-02.csv', 'EW43_UnwghtAvg_2015-11-09.csv', 'EW44_UnwghtAvg_2015-11-16.csv',
                 'EW45_UnwghtAvg_2015-11-23.csv', 'EW46_UnwghtAvg_2015-12-01.csv', 'EW47_UnwghtAvg_2015-12-07.csv',
                 'EW48_UnwghtAvg_2015-12-14.csv', 'EW49_UnwghtAvg_2015-12-21.csv', 'EW50_UnwghtAvg_2015-12-30.csv',
                 'EW51_UnwghtAvg_2016-01-06.csv', 'EW52_UnwghtAvg_2016-01-11.csv']

forecasts1516_all = pd.DataFrame()
for i in forecasts1516:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2015-2016/UnwghtAvg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[df.type=='Bin']
    
    df['submission_date'] = pd.to_datetime(i[15:-4])
    
    df['forecast_date'] = pd.to_datetime(i[15:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1516_all = pd.concat([forecasts1516_all,df])
    
forecasts1516_all['season'] = '2015-2016'
forecasts1516_all['horizon'] = forecasts1516_all['horizon']-1

In [ ]:
forecasts1617 = ['EW01_UnwghtAvg_2017-01-17.csv', 'EW02_UnwghtAvg_2017-01-23.csv', 'EW03_UnwghtAvg_2017-01-30.csv', 
                'EW04_UnwghtAvg_2017-02-06.csv', 'EW05_UnwghtAvg_2017-02-13.csv', 'EW06_UnwghtAvg_2017-02-21.csv',
                'EW07_UnwghtAvg_2017-02-27.csv', 'EW08_UnwghtAvg_2017-03-06.csv', 'EW09_UnwghtAvg_2017-03-14.csv',
                'EW10_UnwghtAvg_2017-03-21.csv', 'EW11_UnwghtAvg_2017-03-28.csv', 'EW12_UnwghtAvg_2017-04-04.csv',
                'EW13_UnwghtAvg_2017-04-11.csv', 'EW14_UnwghtAvg_2017-04-18.csv', 'EW15_UnwghtAvg_2017-04-24.csv',
                'EW16_UnwghtAvg_2017-05-01.csv', 'EW17_UnwghtAvg_2017-05-09.csv', 'EW18_UnwghtAvg_2017-05-15.csv',
                'EW43_UnwghtAvg_2016-11-07.csv', 'EW44_UnwghtAvg_2016-11-14.csv', 'EW45_UnwghtAvg_2016-11-21.csv',
                'EW46_UnwghtAvg_2016-11-28.csv', 'EW47_UnwghtAvg_2016-12-05.csv', 'EW48_UnwghtAvg_2016-12-12.csv',
                'EW49_UnwghtAvg_2016-12-19.csv', 'EW50_UnwghtAvg_2016-12-23.csv', 'EW51_UnwghtAvg_2017-01-02.csv',
                'EW52_UnwghtAvg_2017-01-09.csv']

forecasts1617_all = pd.DataFrame()
for i in forecasts1617:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2016-2017/UnwghtAvg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[15:-4])
    
    df['forecast_date'] = pd.to_datetime(i[15:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100

    
    forecasts1617_all = pd.concat([forecasts1617_all,df])
    
forecasts1617_all['season'] = '2016-2017'
forecasts1617_all['horizon'] = forecasts1617_all['horizon']-1

In [ ]:
forecasts1718 = ['EW01_UnwghtAvg_2018-01-16.csv', 'EW02_UnwghtAvg_2018-01-22.csv', 'EW03_UnwghtAvg_2018-01-29.csv',
                 'EW04_UnwghtAvg_2018-02-05.csv', 'EW05_UnwghtAvg_2018-02-12.csv', 'EW06_UnwghtAvg_2018-02-20.csv',
                 'EW07_UnwghtAvg_2018-02-26.csv', 'EW08_UnwghtAvg_2018-03-05.csv', 'EW09_UnwghtAvg_2018-03-13.csv',
                 'EW10_UnwghtAvg_2018-03-20.csv', 'EW11_UnwghtAvg_2018-03-26.csv', 'EW12_UnwghtAvg_2018-04-03.csv',
                 'EW13_UnwghtAvg_2018-04-10.csv', 'EW14_UnwghtAvg_2018-04-17.csv', 'EW15_UnwghtAvg_2018-04-24.csv',
                 'EW16_UnwghtAvg_2018-05-01.csv', 'EW17_UnwghtAvg_2018-05-08.csv', 'EW18_UnwghtAvg_2018-05-14.csv',
                 'EW43_UnwghtAvg_2017-11-06.csv', 'EW44_UnwghtAvg_2017-11-13.csv', 'EW45_UnwghtAvg_2017-11-20.csv',
                 'EW46_UnwghtAvg_2017-11-29.csv', 'EW47_UnwghtAvg_2017-12-04.csv', 'EW48_UnwghtAvg_2017-12-11.csv',
                 'EW49_UnwghtAvg_2017-12-18.csv', 'EW50_UnwghtAvg_2017-12-27.csv', 'EW51_UnwghtAvg_2018-01-03.csv',
                 'EW52_UnwghtAvg_2018-01-08.csv']

forecasts1718_all = pd.DataFrame()
for i in forecasts1718:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2017-2018/UnwghtAvg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[15:-4])
    
    df['forecast_date'] = pd.to_datetime(i[15:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1718_all = pd.concat([forecasts1718_all,df])
    
forecasts1718_all['season'] = '2017-2018'
forecasts1718_all['horizon'] = forecasts1718_all['horizon']-1

In [ ]:
forecasts1819 = ['EW01_UnwghtAvg_2019-01-15.csv', 'EW02_UnwghtAvg_2019-01-22.csv', 'EW03_UnwghtAvg_2019-01-29.csv',
                 'EW04_UnwghtAvg_2019-02-05.csv', 'EW05_UnwghtAvg_2019-02-12.csv', 'EW06_UnwghtAvg_2019-02-20.csv',
                 'EW07_UnwghtAvg_2019-02-26.csv', 'EW08_UnwghtAvg_2019-03-05.csv', 'EW09_UnwghtAvg_2019-03-12.csv',
                 'EW10_UnwghtAvg_2019-03-19.csv', 'EW11_UnwghtAvg_2019-03-26.csv', 'EW12_UnwghtAvg_2019-04-02.csv',
                 'EW13_UnwghtAvg_2019-04-09.csv', 'EW14_UnwghtAvg_2019-04-16.csv', 'EW15_UnwghtAvg_2019-04-23.csv',
                 'EW16_UnwghtAvg_2019-04-30.csv', 'EW17_UnwghtAvg_2019-05-07.csv', 'EW18_UnwghtAvg_2019-05-14.csv',
                 'EW42_UnwghtAvg_2018-10-29.csv', 'EW43_UnwghtAvg_2018-11-04.csv', 'EW44_UnwghtAvg_2018-11-11.csv',
                 'EW45_UnwghtAvg_2018-11-20.csv', 'EW46_UnwghtAvg_2018-11-27.csv', 'EW47_UnwghtAvg_2018-12-04.csv',
                 'EW48_UnwghtAvg_2018-12-11.csv', 'EW49_UnwghtAvg_2018-12-18.csv', 'EW50-UnwghtAvg-2018-12-25.csv',
                 'EW51_UnwghtAvg_2019-01-01.csv', 'EW52_UnwghtAvg_2019-01-08.csv']


forecasts1819_all = pd.DataFrame()
for i in forecasts1819:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2018-2019/UnwghtAvg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[15:-4])
    
    df['forecast_date'] = pd.to_datetime(i[15:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1819_all = pd.concat([forecasts1819_all,df])
    
forecasts1819_all['season'] = '2018-2019'
forecasts1819_all['horizon'] = forecasts1819_all['horizon']-1

In [ ]:
forecasts1920 =['EW01_UnwghtAvg_2020-01-14.csv', 'EW02_UnwghtAvg_2020-01-21.csv', 'EW03_UnwghtAvg_2020-01-28.csv',
                'EW04_UnwghtAvg_2020-02-04.csv', 'EW05_UnwghtAvg_2020-02-11.csv', 'EW06_UnwghtAvg_2020-02-19.csv',
                'EW07_UnwghtAvg_2020-02-25.csv', 'EW08_UnwghtAvg_2020-03-04.csv', 'EW09_UnwghtAvg_2020-03-10.csv',
                'EW42_UnwghtAvg_2019-10-29.csv', 'EW43_UnwghtAvg_2019-11-05.csv', 'EW44_UnwghtAvg_2019-11-13.csv',
                'EW45_UnwghtAvg_2019-11-19.csv', 'EW46_UnwghtAvg_2019-11-26.csv', 'EW47_UnwghtAvg_2019-12-04.csv',
                'EW48_UnwghtAvg_2019-12-10.csv', 'EW49_UnwghtAvg_2019-12-17.csv', 'EW50_UnwghtAvg_2019-12-24.csv',
                'EW51_UnwghtAvg_2019-12-31.csv', 'EW52_UnwghtAvg_2020-01-07.csv']

forecasts1920_all = pd.DataFrame()
for i in forecasts1920:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2019-2020/UnwghtAvg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[15:-4])
    
    df['forecast_date'] = pd.to_datetime(i[15:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1920_all = pd.concat([forecasts1920_all,df])
    
forecasts1920_all['season'] = '2019-2020'
forecasts1920_all['horizon'] = forecasts1920_all['horizon']-1

In [ ]:
forecasts_all = pd.concat([forecasts1516_all, forecasts1617_all, forecasts1718_all, forecasts1819_all, 
                           forecasts1920_all])
forecasts_all.to_csv('dat/ILIforecasts_all.csv', index=False)

# ILI hub baseline forecasts (Hist-Avg)

In [ ]:
forecasts1516 = ['EW01_Hist-Avg_2016-01-18.csv', 'EW02_Hist-Avg_2016-01-25.csv', 'EW03_Hist-Avg_2016-02-01.csv',
                 'EW04_Hist-Avg_2016-02-08.csv', 'EW05_Hist-Avg_2016-02-15.csv', 'EW06_Hist-Avg_2016-02-22.csv',
                 'EW07_Hist-Avg_2016-02-29.csv', 'EW08_Hist-Avg_2016-03-07.csv', 'EW09_Hist-Avg_2016-03-14.csv',
                 'EW10_Hist-Avg_2016-03-21.csv', 'EW11_Hist-Avg_2016-03-28.csv', 'EW12_Hist-Avg_2016-04-04.csv',
                 'EW13_Hist-Avg_2016-04-11.csv', 'EW14_Hist-Avg_2016-04-18.csv', 'EW15_Hist-Avg_2016-04-25.csv',
                 'EW16_Hist-Avg_2016-05-02.csv', 'EW17_Hist-Avg_2016-05-09.csv', 'EW18_Hist-Avg_2016-05-16.csv',
                 'EW42_Hist-Avg_2015-11-02.csv', 'EW43_Hist-Avg_2015-11-09.csv', 'EW44_Hist-Avg_2015-11-16.csv',
                 'EW45_Hist-Avg_2015-11-23.csv', 'EW46_Hist-Avg_2015-12-02.csv', 'EW47_Hist-Avg_2015-12-07.csv',
                 'EW48_Hist-Avg_2015-12-14.csv', 'EW49_Hist-Avg_2015-12-21.csv', 'EW50_Hist-Avg_2015-12-30.csv',
                 'EW51_Hist-Avg_2016-01-06.csv', 'EW52_Hist-Avg_2016-01-11.csv']

forecasts1516_all = pd.DataFrame()
for i in forecasts1516:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2015-2016/Hist-Avg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[df.type=='Bin']
    
    df['submission_date'] = pd.to_datetime(i[14:-4])
    
    df['forecast_date'] = pd.to_datetime(i[14:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1516_all = pd.concat([forecasts1516_all,df])
    
forecasts1516_all['season'] = '2015-2016'
forecasts1516_all['horizon'] = forecasts1516_all['horizon']-1

In [ ]:
forecasts1617 = ['EW01-Hist-Avg-2017-01-17.csv', 'EW02-Hist-Avg-2017-01-23.csv', 'EW03-Hist-Avg-2017-01-30.csv', 
                'EW04-Hist-Avg-2017-02-06.csv', 'EW05-Hist-Avg-2017-02-13.csv', 'EW06-Hist-Avg-2017-02-21.csv',
                'EW07-Hist-Avg-2017-02-27.csv', 'EW08_Hist-Avg_2017-03-06.csv', 'EW09-Hist-Avg-2017-03-14.csv',
                'EW10_Hist-Avg_2017-03-20.csv', 'EW11_Hist-Avg_2017-03-27.csv', 'EW12_Hist-Avg_2017-04-03.csv',
                'EW13_Hist-Avg_2017-04-10.csv', 'EW14_Hist-Avg_2017-04-17.csv', 'EW15_Hist-Avg_2017-04-24.csv',
                'EW16_Hist-Avg_2017-05-01.csv', 'EW17_Hist-Avg_2017-05-08.csv', 'EW18_Hist-Avg_2017-05-15.csv',
                'EW43_Hist-Avg_2016-11-07.csv', 'EW44_Hist-Avg_2016-11-14.csv', 'EW45-Hist-Avg-2016-11-21.csv',
                'EW46_Hist-Avg_2016-11-30.csv', 'EW47_Hist-Avg_2016-12-05.csv', 'EW48_Hist-Avg_2016-12-12.csv',
                'EW49_Hist-Avg_2016-12-19.csv', 'EW50_Hist-Avg_2016-12-28.csv', 'EW51_Hist-Avg_2017-01-04.csv',
                'EW52-Hist-Avg-2017-01-09.csv']

forecasts1617_all = pd.DataFrame()
for i in forecasts1617:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2016-2017/Hist-Avg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[14:-4])
    
    df['forecast_date'] = pd.to_datetime(i[14:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100

    
    forecasts1617_all = pd.concat([forecasts1617_all,df])
    
forecasts1617_all['season'] = '2016-2017'
forecasts1617_all['horizon'] = forecasts1617_all['horizon']-1

In [ ]:
forecasts1718 = ['EW01-Hist-Avg-2018-01-16.csv', 'EW02-Hist-Avg-2018-01-22.csv', 'EW03-Hist-Avg-2018-01-29.csv',
                 'EW04-Hist-Avg-2018-02-05.csv', 'EW05-Hist-Avg-2018-02-12.csv', 'EW06-Hist-Avg-2018-02-20.csv',
                 'EW07-Hist-Avg-2018-02-26.csv', 'EW08-Hist-Avg-2018-03-05.csv', 'EW09-Hist-Avg-2018-03-13.csv',
                 'EW10-Hist-Avg-2018-03-20.csv', 'EW11-Hist-Avg-2018-03-27.csv', 'EW12-Hist-Avg-2018-04-03.csv',
                 'EW13-Hist-Avg-2018-04-10.csv', 'EW14-Hist-Avg-2018-04-17.csv', 'EW15-Hist-Avg-2018-04-24.csv',
                 'EW16-Hist-Avg-2018-05-01.csv', 'EW17-Hist-Avg-2018-05-08.csv', 'EW18-Hist-Avg-2018-05-15.csv',
                 'EW43-Hist-Avg-2017-11-06.csv', 'EW44-Hist-Avg-2017-11-13.csv', 'EW45-Hist-Avg-2017-11-20.csv',
                 'EW46-Hist-Avg-2017-11-29.csv', 'EW47-Hist-Avg-2017-12-04.csv', 'EW48-Hist-Avg-2017-12-11.csv',
                 'EW49-Hist-Avg-2017-12-18.csv', 'EW50-Hist-Avg-2017-12-27.csv', 'EW51-Hist-Avg-2018-01-03.csv',
                 'EW52-Hist-Avg-2018-01-08.csv']

forecasts1718_all = pd.DataFrame()
for i in forecasts1718:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2017-2018/Hist-Avg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[14:-4])
    
    df['forecast_date'] = pd.to_datetime(i[14:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1718_all = pd.concat([forecasts1718_all,df])
    
forecasts1718_all['season'] = '2017-2018'
forecasts1718_all['horizon'] = forecasts1718_all['horizon']-1

In [ ]:
forecasts1819 = ['EW01-Hist-Avg-2019-01-15.csv', 'EW02-Hist-Avg-2019-01-22.csv', 'EW03-Hist-Avg-2019-01-29.csv',
                 'EW04-Hist-Avg-2019-02-05.csv', 'EW05-Hist-Avg-2019-02-12.csv', 'EW06-Hist-Avg-2019-02-20.csv',
                 'EW07-Hist-Avg-2019-02-26.csv', 'EW08_Hist-Avg_2019-03-04.csv', 'EW09-Hist-Avg-2019-03-12.csv',
                 'EW10-Hist-Avg-2019-03-19.csv', 'EW11-Hist-Avg-2019-03-26.csv', 'EW12-Hist-Avg-2019-04-02.csv',
                 'EW13-Hist-Avg-2019-04-09.csv', 'EW14-Hist-Avg-2019-04-16.csv', 'EW15-Hist-Avg-2019-04-23.csv',
                 'EW16-Hist-Avg-2019-04-30.csv', 'EW17-Hist-Avg-2019-05-07.csv', 'EW18-Hist-Avg-2019-05-14.csv',
                 'EW42_Hist-Avg_2018-10-29.csv', 'EW43_Hist-Avg_2018-11-05.csv', 'EW44_Hist-Avg_2018-11-12.csv',
                 'EW45_Hist-Avg_2018-11-19.csv', 'EW46-Hist-Avg-2018-11-27.csv', 'EW47-Hist-Avg-2018-12-04.csv',
                 'EW48-Hist-Avg-2018-12-11.csv', 'EW49-Hist-Avg-2018-12-18.csv', 'EW50-Hist-Avg-2018-12-25.csv',
                 'EW51-Hist-Avg-2019-01-01.csv', 'EW52-Hist-Avg-2019-01-08.csv']


forecasts1819_all = pd.DataFrame()
for i in forecasts1819:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2018-2019/Hist-Avg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[14:-4])
    
    df['forecast_date'] = pd.to_datetime(i[14:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1819_all = pd.concat([forecasts1819_all,df])
    
forecasts1819_all['season'] = '2018-2019'
forecasts1819_all['horizon'] = forecasts1819_all['horizon']-1

In [ ]:
forecasts1920 =['EW01_Hist-Avg_2020-01-13.csv', 'EW02-Hist-Avg-2020-01-22.csv', 'EW03-Hist-Avg-2020-01-28.csv',
                'EW05-Hist-Avg-2020-02-11.csv', 'EW06-Hist-Avg-2020-02-19.csv',
                'EW07-Hist-Avg-2020-02-25.csv',  'EW09-Hist-Avg-2020-03-10.csv',
                'EW42-Hist-Avg-2019-10-29.csv', 'EW43-Hist-Avg-2019-11-05.csv', 'EW44-Hist-Avg-2019-11-13.csv',
                'EW45-Hist-Avg-2019-11-19.csv', 'EW46_Hist-Avg_2019-11-25.csv', 'EW47_Hist-Avg_2019-12-02.csv',
                'EW48-Hist-Avg-2019-12-10.csv', 'EW49-Hist-Avg-2019-12-17.csv', 'EW50-Hist-Avg-2019-12-24.csv',
                'EW51-Hist-Avg-2019-12-31.csv', 'EW52-Hist-Avg-2020-01-07.csv']

forecasts1920_all = pd.DataFrame()
for i in forecasts1920:
    df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2019-2020/Hist-Avg/'+i)
    df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                           'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
    df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]
    
    df = df[(df.type=='Bin')  & (df.unit == "percent")]
    
    df['submission_date'] = pd.to_datetime(i[14:-4])
    
    df['forecast_date'] = pd.to_datetime(i[14:-4])
    df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

    df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)
        
    df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())
    
    df['bin_start_incl'] = df['bin_start_incl'].astype(float)
    df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)
    
    df['bin_start_incl'] = df['bin_start_incl']/100
    df['bin_end_notincl'] = df['bin_end_notincl']/100
    
    forecasts1920_all = pd.concat([forecasts1920_all,df])
    
forecasts1920_all['season'] = '2019-2020'
forecasts1920_all['horizon'] = forecasts1920_all['horizon']-1

In [ ]:
forecasts_all = pd.concat([forecasts1516_all, forecasts1617_all, forecasts1718_all, forecasts1819_all, 
                           forecasts1920_all])

forecasts_all.to_csv('dat/ILIforecasts_Hist-Avg.csv')

# ILI team forecasts

In [ ]:
models = ['4Sight', 'ARETE', 'CU1', 'CU2', 'Delphi-Archefilter', 'Delphi-Epicast', 'Delphi-Stat', 'ISU',
          'JL', 'KBSI1', 'KOT', 'NEU', 'PSI', 'UMN']

forecasts1516_all = pd.DataFrame()
for model in models:
    forecasts1516 = [f'EW01_{model}_2016-01-18.csv', f'EW02_{model}_2016-01-25.csv', f'EW03_{model}_2016-02-01.csv',
                 f'EW04_{model}_2016-02-08.csv', f'EW05_{model}_2016-02-15.csv', f'EW06_{model}_2016-02-22.csv',
                 f'EW07_{model}_2016-02-29.csv', f'EW08_{model}_2016-03-07.csv', f'EW09_{model}_2016-03-14.csv',
                 f'EW10_{model}_2016-03-21.csv', f'EW11_{model}_2016-03-28.csv', f'EW12_{model}_2016-04-04.csv',
                 f'EW13_{model}_2016-04-11.csv', f'EW14_{model}_2016-04-18.csv', f'EW15_{model}_2016-04-25.csv',
                 f'EW16_{model}_2016-05-02.csv', f'EW17_{model}_2016-05-09.csv', f'EW18_{model}_2016-05-16.csv',
                 f'EW42_{model}_2015-11-02.csv', f'EW43_{model}_2015-11-09.csv', f'EW44_{model}_2015-11-16.csv',
                 f'EW45_{model}_2015-11-23.csv', f'EW46_{model}_2015-12-01.csv', f'EW46_{model}_2015-11-30.csv',
                 f'EW47_{model}_2015-12-07.csv', f'EW48_{model}_2015-12-14.csv', f'EW49_{model}_2015-12-21.csv',
                 f'EW49_{model}_2015-12-18.csv', f'EW50_{model}_2015-12-30.csv', f'EW51_{model}_2016-01-06.csv', 
                 f'EW52_{model}_2016-01-11.csv', f'EW52_{model}_2016-01-08.csv', ]

    for i in forecasts1516:
        try:
            df = pd.read_csv(f'https://raw.githubusercontent.com/cdcepi/FluSight-forecasts/refs/heads/master/2015-2016/{model}/'+i)
            df = df.rename(columns={'Location':'location','Target':'target','Type':'type','Bin_start_incl':'bin_start_incl',
                                   'Bin_end_notincl':'bin_end_notincl','Value':'value','Unit':'unit'})
            df = df[df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])]

            df = df[df.type=='Bin']

            df['submission_date'] = pd.to_datetime(i[-14:-4])

            df['forecast_date'] = pd.to_datetime(i[-14:-4])
            df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

            df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)

            df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
            df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
            df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())

            df['bin_start_incl'] = df['bin_start_incl'].astype(float)
            df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)

            df['bin_start_incl'] = df['bin_start_incl']/100
            df['bin_end_notincl'] = df['bin_end_notincl']/100

            df['Model'] = model

            forecasts1516_all = pd.concat([forecasts1516_all,df])
        except:
            print(f'No forecast for {model} at {i}')

    forecasts1516_all['season'] = '2015-2016'

forecasts1516_all['horizon'] = forecasts1516_all['horizon']-1

In [ ]:
def load_github_folder_as_df(owner, repo, folder_path, branch="master"):
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{folder_path}?ref={branch}"
    
    response = requests.get(api_url)
    response.raise_for_status()
    files = response.json()

    dfs = []
    for file in files:
        if file["type"] == "file" and file["name"].endswith(".csv"):
            print(f"Loading {file['name']}...")
            raw = requests.get(file["download_url"])
            raw.raise_for_status()
            df = pd.read_csv(io.StringIO(raw.text))
            
            
            df.columns = df.columns.str.lower()
            
            df = df[(df.type=='Bin') & (df.target.isin(['1 wk ahead', '2 wk ahead', '3 wk ahead', '4 wk ahead'])) &\
                   (df.unit == "percent")]
            
            if file["name"]=='EW46-ARETE-11-27-2017.csv':
                df['submission_date'] = pd.to_datetime('2017-11-27')
                df['forecast_date'] = pd.to_datetime('2017-11-27')
            elif file["name"] == 'EW09-2018-ReichLab_KoT.csv':
                df['submission_date'] = pd.to_datetime('2018-03-13')
                df['forecast_date'] = pd.to_datetime('2018-03-13')
            elif file["name"] == 'EW43-BioFire-2018-11-5.csv':
                df['submission_date'] = pd.to_datetime('2018-11-05')
                df['forecast_date'] = pd.to_datetime('2018-11-05')
            elif file["name"] == 'EW18-2019-ReichLab_KoT.csv':
                df['submission_date'] = pd.to_datetime('2019-05-14')
                df['forecast_date'] = pd.to_datetime('2019-05-14')
            elif file["name"] == 'EW43-2018-ReichLab_KoT_stable.csv':
                df['submission_date'] = pd.to_datetime('2018-11-05')
                df['forecast_date'] = pd.to_datetime('2018-11-05')
            elif file["name"]== 'EW52-2018-ReichLab_KoT_stable.csv':
                df['submission_date'] = pd.to_datetime('2019-01-08')
                df['forecast_date'] = pd.to_datetime('2019-01-08')
            elif file["name"] == 'EW43-2018-ReichLab_KoT_dev.csv':
                df['submission_date'] = pd.to_datetime('2018-11-05')
                df['forecast_date'] = pd.to_datetime('2018-11-05')
            elif file["name"] == 'EW44-2018-ReichLab_KoT_dev.csv':
                df['submission_date'] = pd.to_datetime('2018-11-12')
                df['forecast_date'] = pd.to_datetime('2018-11-12')
            elif file["name"] == 'EW52-2018-ReichLab_KoT_dev.csv':
                df['submission_date'] = pd.to_datetime('2019-01-08')
                df['forecast_date'] = pd.to_datetime('2019-01-08')
            elif file["name"] == 'EW43-2018-LANL_DBMplus.csv':
                df['submission_date'] = pd.to_datetime('2018-11-05')
                df['forecast_date'] = pd.to_datetime('2018-11-05')
            elif file["name"] == 'EW43-2018-LANL_Dante.csv':
                df['submission_date'] = pd.to_datetime('2018-11-05')
                df['forecast_date'] = pd.to_datetime('2018-11-05')
            elif file["name"] == 'EW52-GSU_extLog-2019-1-6_LATE.csv':
                df['submission_date'] = pd.to_datetime('2020-01-06')
                df['forecast_date'] = pd.to_datetime('2020-01-06')
            elif file["name"] == 'EW01-KoT-adaptive-2020-1-18.csv':
                df['submission_date'] = pd.to_datetime('2020-01-14')
                df['forecast_date'] = pd.to_datetime('2020-01-14')
            elif file["name"] == 'EW02-KoT-adaptive-2020-2-5.csv':
                df['submission_date'] = pd.to_datetime('2020-01-21')
                df['forecast_date'] = pd.to_datetime('2020-01-21')
            elif file["name"] =='EW03-KoT-adaptive-2020-2-5.csv':
                df['submission_date'] = pd.to_datetime('2020-01-28')
                df['forecast_date'] = pd.to_datetime('2020-01-28')
            elif file["name"] == 'EW43-KoT-adaptive-2019-11-6.csv':
                df['submission_date'] = pd.to_datetime('2019-11-06')
                df['forecast_date'] = pd.to_datetime('2019-11-06')
            elif file["name"] == 'EW46-KoT-adaptive-2019-12-2.csv':
                df['submission_date'] = pd.to_datetime('2019-12-02')
                df['forecast_date'] = pd.to_datetime('2019-12-02')
            elif file["name"] == 'EW47-KoT-adaptive-2019-12-9.csv':
                df['submission_date'] = pd.to_datetime('2019-12-09')
                df['forecast_date'] = pd.to_datetime('2019-12-09')
            elif file["name"] == 'EW50-KoT-adaptive-2020-1-4.csv':
                df['submission_date'] = pd.to_datetime('2019-12-24')
                df['forecast_date'] = pd.to_datetime('2019-12-24')
            elif file["name"] == 'EW51-KoT-adaptive-2020-1-4.csv':
                df['submission_date'] = pd.to_datetime('2019-12-31')
                df['forecast_date'] = pd.to_datetime('2019-12-31')
            elif file["name"] == 'EW52-KoT-adaptive-2020-1-18.csv':
                df['submission_date'] = pd.to_datetime('2020-01-07')
                df['forecast_date'] = pd.to_datetime('2020-01-07')
            else:
                df['submission_date'] = pd.to_datetime(re.search(r"\d{4}-\d{2}-\d{2}", file["name"]).group())
                df['forecast_date'] = pd.to_datetime(re.search(r"\d{4}-\d{2}-\d{2}", file["name"]).group())

            
            df['forecast_date'] = df['forecast_date'].apply(lambda x: Week.fromdate(x).startdate() + pd.Timedelta(days=2))

            df['horizon'] = df['target'].apply(lambda key: int(key[0])-1)

            df['target_end_date'] =  df['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
            df['target_end_date'] =  df['target_end_date'] + df['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
            df['target_end_date'] =  df['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())

            df['bin_start_incl'] = df['bin_start_incl'].astype(float)
            df['bin_end_notincl'] = df['bin_end_notincl'].astype(float)

            df['bin_start_incl'] = df['bin_start_incl']/100
            df['bin_end_notincl'] = df['bin_end_notincl']/100

            
            dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    #print(f"\nDone! {len(dfs)} files loaded, {len(combined)} rows total.")
    return combined



In [ ]:
models = ['4Sight', 'CU1', 'CU2', 'CU3', 'CU4', 'Delphi-Epicast', 'Delphi-Stat', 'FORSEA', 'GHRI', 'Harvard', 
          'HumNat', 'HumNat2', 'ICS', 'ISU', 'KBSI', 'KOT-Dev', 'KOT-Stable', 'LANL', 'NEU', 'PSI', 'TeamA', 
          'TeamB', 'TeamC', 'TeamD', 'TeamE', 'UoM-DSTG', 'Yale1', 'Yale2']

forecasts1617_all = pd.DataFrame()
for model in models:
    df = load_github_folder_as_df(
        owner="cdcepi",
        repo="FluSight-forecasts",
        folder_path=f"2016-2017/{model}",
        branch="master",
    )

    df['Model'] = model
    df['season'] = '2016-2017'
    
    forecasts1617_all = pd.concat([forecasts1617_all, df])
forecasts1617_all['horizon'] = forecasts1617_all['horizon']-1

In [ ]:
models = ['02115-emmsa', '02115_emm', '02115_emms', '4Sight', '4Sight2', 'ARETE', 'CU-Network', 'CU-SEL',
          'CU_Puffins', 'CU_Vixen', 'Delphi-Epicast', 'Delphi-Stat', 'HumNat', 'ISU', 'KBSI', 'KPWHRI',
          'Kernel of Truth', 'LANL-DBM', 'LANL-DBMplus', 'NEU-GLEAM', 'PPFST Crowd', 'PredSci', 'Team A',
          'Team_GA', 'Tulane', 'UA Epiflu', 'UA I', 'UCSF3', 'UMNSpl', 'YaleModel', 'uom']

forecasts1718_all = pd.DataFrame()
for model in models:
    df = load_github_folder_as_df(
        owner="cdcepi",
        repo="FluSight-forecasts",
        folder_path=f"2017-2018/{model}",
        branch="master",
    )

    df['Model'] = model
    df['season'] = '2017-2018'
    
    forecasts1718_all = pd.concat([forecasts1718_all, df])
    
forecasts1718_all['horizon'] = forecasts1718_all['horizon']-1

In [ ]:
models = ['ARGO', 'BioFire FLI', 'CEID', 'CU_Network', 'CU_Puffins', 'CU_Sel', 'CU_Vixen', 'Delphi-Epicast',
          'Delphi-Epicast2', 'Delphi-Stat', 'EpiDeep', 'FluOutlook-Mech', 'FluOutlook-MechArgo',
          'FluOutlook-MechAug', 'FluX', 'GH-TeamA', 'GH-TeamB', 'ISU', 'KBSI', 'KPWHRI', 'KoT', 'KoT_dev',
          'LANL-DBMplus', 'LANL-Dante', 'PPFST', 'PPFST2', 'PSI-e', 'PSI-s', 'Protea-Cheetah', 'Protea-Springbok',
          'TeamAnonymous', 'Tulane2018', 'UA EpiFlu', 'UA-EpiCos', 'UMNSpl', 'UniMelb', 'YaleModel']

forecasts1819_all = pd.DataFrame()
for model in models:
    df = load_github_folder_as_df(
        owner="cdcepi",
        repo="FluSight-forecasts",
        folder_path=f"2018-2019/{model}",
        branch="master",
    )

    df['Model'] = model
    df['season'] = '2018-2019'
    
    forecasts1819_all = pd.concat([forecasts1819_all, df])
    
forecasts1819_all['horizon'] = forecasts1819_all['horizon']-1

In [ ]:
models = ['CEID', 'CU_Albatross', 'CU_Puffins', 'CU_Ravens', 'CU_Vixen', 'Delphi-Epicast-Mturk', 'Delphi-Epicast',
          'Delphi-Stat', 'Draft', 'EpiDeep', 'EvoEpiFlu', 'EvoEpiFlu_cluster', 'Fire-FLI', 'FluOutlook-MechAug',
          'FluOutlook-MechAug2', 'FluX_ARLR', 'FluX_LSTM', 'GSU_extLog', 'Guidehouse_A', 'KISTI_CSB_26weeks',
          'KISTI_JHKIM', 'KPWHRI', 'Kernel of Truth', 'KoT-GPthingy', 'KoT-adaptive', 'LANL_DBMplus',
          'LANL_Danteplus', 'PPFST', 'PSI-e', 'PSI-s', 'Protea-Cheetah', 'Protea-Springbok', 'TeamAnonymous',
          'UA_EpiCos', 'UMNSpl']

forecasts1920_all = pd.DataFrame()
for model in models:
    df = load_github_folder_as_df(
        owner="cdcepi",
        repo="FluSight-forecasts",
        folder_path=f"2019-2020/{model}",
        branch="master",
    )

    df['Model'] = model
    df['season'] = '2019-2020'
    
    forecasts1920_all = pd.concat([forecasts1920_all, df])
    
forecasts1920_all['horizon'] = forecasts1920_all['horizon']-1

In [ ]:
os.mkdir('dat/ILI team forecasts')

forecasts1516_all.to_csv('dat/ILI team forecasts/ILIforecasts_teams_2015-16.csv.gz', index=False, compression='gzip')
forecasts1617_all.to_csv('dat/ILI team forecasts/ILIforecasts_teams_2016-17.csv.gz', index=False, compression='gzip')
forecasts1718_all.to_csv('dat/ILI team forecasts/ILIforecasts_teams_2017-18.csv.gz', index=False, compression='gzip')
forecasts1819_all.to_csv('dat/ILI team forecasts/ILIforecasts_teams_2018-19.csv.gz', index=False, compression='gzip')
forecasts1920_all.to_csv('dat/ILI team forecasts/ILIforecasts_teams_2019-20.csv.gz', index=False, compression='gzip')